In [ ]:
from nrem_sc.constants import PROCESSED_DATA_PATH, INTERIM_DATA_PATH
import numpy as np
import pandas as pd
import pynapple as nap
import matplotlib.pyplot as plt
import seaborn as sns

📂 ucsf
├── 📂 external
├── 📂 interim
│   ├── hd_burst_epochs.npz     |        IntervalSet
│   └── hd_pop_zrate.npz        |        Tsd
├── 📂 processed
│   ├── active_wake.npz         |        IntervalSet
│   ├── angle_openfield.npz     |        Tsd
│   ├── hd_spikes_openfield.npz         |        TsdFrame
│   ├── hd_spikes_total.npz     |        TsGroup
│   ├── position_neck.npz       |        TsdFrame
│   ├── pupil_full_data.npz     |        TsdFrame
│   ├── pupil_full_normalized.npz       |        TsdFrame
│   ├── pupil_nrem.npz  |        TsdFrame
│   ├── pupil_nrem_normalized.npz       |        TsdFrame
│   ├── sleep.npz       |        IntervalSet
│   ├── spikes_shank_1.npz      |        TsGroup
│   ├── spikes_shank_2.npz      |        TsGroup
│   ├── spikes_shank_3.npz      |        TsGroup
│   └── turn_spikes.npz         |        TsGroup
└── 📂 raw

In [2]:
sleep_states    = nap.load_file(PROCESSED_DATA_PATH / "sleep.npz")
pupil_data      = nap.load_file(PROCESSED_DATA_PATH / "pupil_nrem_normalized.npz")
hd_spikes       = nap.load_file(PROCESSED_DATA_PATH / "hd_spikes_total.npz")
turn_spikes     = nap.load_file(PROCESSED_DATA_PATH / "turn_spikes.npz")

Approach: Detect population level bursts (i.e local maxima) during NREM sleep

In [3]:
nrem = sleep_states[sleep_states['state'] == 'nrem']
nrem

index    start    end      state
0        2048.0   2133.0   nrem
1        2143.0   2219.0   nrem
2        2241.0   2263.0   nrem
3        2295.0   2373.0   nrem
4        3002.0   3033.0   nrem
5        3050.0   3089.0   nrem
6        3098.0   3121.0   nrem
...      ...      ...      ...
304      76749.0  76808.0  nrem
305      76824.0  76915.0  nrem
306      77144.0  77163.0  nrem
307      77172.0  77280.0  nrem
308      77309.0  77492.0  nrem
309      79391.0  79578.0  nrem
310      80587.0  80626.0  nrem
shape: (311, 2), time unit: sec.

In [20]:
# Count 20 ms bins
counts = hd_spikes.count(0.02, ep=nrem)

# Compute smoothed population rate
pop_rate = counts.sum(axis=1).smooth(std=0.05)

# Z-score (relative to NREM)
z_rate = (pop_rate - pop_rate.mean()) / pop_rate.std()


Time (s)
----------  ----------
2048.01     -0.626807
2048.03     -0.25103
2048.05      0.0520736
2048.07      0.214661
2048.09      0.240112
2048.11      0.301184
2048.13      0.721374
...
80625.87    -0.606122
80625.89    -0.746472
80625.91    -0.855648
80625.93    -0.920578
80625.95    -0.973164
80625.97    -1.0562
80625.99    -1.18702
dtype: float64, shape: (1870600,)

In [25]:
# Detect bursts
burst_epochs = z_rate.threshold(2).time_support
burst_epochs = burst_epochs.drop_short_intervals(0.04)

In [43]:
z_rate.save(INTERIM_DATA_PATH / "hd_pop_zrate.npz")
burst_epochs.save(INTERIM_DATA_PATH / "hd_burst_epochs.npz")

In [ ]:
grp = turn_spikes.getby_category('turn_mode')
cw, ccw = grp['cw'], grp['ccw']

Index    rate      turn_mode
-------  --------  -----------
23       1.43137   cw
24       0.46984   cw
84       3.01458   cw
86       0.38423   cw
93       2.59679   cw
95       0.40564   cw
106      3.61753   cw
...      ...       ...
112      5.58677   cw
118      4.34237   cw
124      10.06167  cw
205      2.80439   cw
216      1.27919   cw
218      2.59453   cw
220      0.58625   cw